## SRP056883 / GSE67587

**paper:** [PMID: 26430061](https://pmc.ncbi.nlm.nih.gov/articles/PMC4684695/) - Expression Divergence of Chemosensory Genes between Drosophila sechellia and Its Sibling Species and Its Implications for Host Shift, 2015

**date, curator:** 2026-07-15, Sara Carsanaro

**notes**
* data from this paper is also in SRP057153 and SRP057154
* updated strain to Rob3c / Tucson 14021-0248.25 per methods

### annotation summary

In [26]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Adult antenna,UBERON:0000972,antenna,perfect match


In [27]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,within 1hr of eclosion,UBERON:0000066,fully formed stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP056883"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX978699,SRP056883,Illumina HiSeq 2000,SRS895369,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650073,Adult antenna,within 1hr of eclosion,,,,F,,"Tuson, #14021-0248-30",7238,,,,,,Dsec female TW rep3,"SAMN03460236,GSM1650073",,,,,,,,,,,15/07/2026,"About 300-700 pairs of fly antennae of each sex from each species were collected for total RNA isolation. The antennae resected each day were preserved in ~ 50-100 ul Trizol (Life Technology) and stored at -80 °C before further processing. Right before the RNA isolation, we spun down antennae preserved in Trizol at the max speed for 3 min at 4 °C and put each sample into one MagNA Lyser Green Beads tube (Roche). The RNA isolation protocol we developed combined steps from conventional Trizol extraction and RNeasy Micro Elute Kit (QIAGEN, Inc.) with some modifications to obtain high yield and quality of total RNA. Fly antennae were disrupted with MagNA Lyser (Roche) at 7000 rpm for 15 seconds each time and repeated for 4-5 times until the tissue was almost invisible. After 15 sec, we cooled down the lysate on ice for 10 sec to prevent RNA degradation. Tissue lysate was transferred to a new RNase free Eppendorf tube. We used about 400 ul Trizol to rinse the beads of each sample and combine this 400 ul Trizol with the tissue lysate for the RNA isolation. Based on the suggestion of the Trizol standard protocol, 200 microliters of chloroform per microliter of tissue lysate were added to the lysate and mixed well by shaking vigorously for 15 sec and set 2-3 min at room temperature. The aqueous phase was separated by centrifuging the lysate at the maximum speed at 4 °C for 15 min and carefully transferred to a new Eppendorf tube after centrifuging. We added 1 volume of 5 µl Carrier RNA (QIAGEN, Inc.) and one microliter of 70% EtOH to the aqueous phase and mixed well by inverting the tubes carefully. To obtain greater amount of RNA, we kept the samples at -80 °C for overnight to aid in RNA precipitation. RNA isolation followed manufacturer’s protocol of the RNeasy Micro Elute Kit with increased volumes (700 ul) of RPE buffer and 80% ethanol to wash RNA. Genomic DNA was removed by applying on-column DNase I treatment at room temperature for 15 min. For paired-end mRNA-seq library preparation, we used Illumina TrueSeq kits. A total of 10 µg total RNA was used for mRNA enrichment by oligo-dT beads followed by cation-catalyzed fragmentation for 4 minutes at 94˚C. The mRNA fragments were then converted into double stranded cDNA by random priming followed by end repair. The fragmented cDNAs from each sex of each species were then ligated to the barcoded paired-end adaptors and subjected to 15 cycles of PCR amplification and purified by Ampure beads (Beckman Agencourt). The absolute concentrations of the libraries were determined by Qubit fluorometry (Invitrogen) and BioAnalyzer High Sensitivity DNA Kit (Agilent). We combined barcoded cDNA from 6 samples and loaded them in 3 sequencing lanes of flow cell, and paired-end 2*100nt sequencing was conducted on Illumina HiSeq2000.",,,,Adult antenna,,Adult,,TRANSCRIPTOMIC,cDNA
1,SRX978698,SRP056883,Illumina HiSeq 2000,SRS895370,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650072,Adult antenna,within 1hr of eclosion,,,,F,,"Tuson, #14021-0248-29",7238,,,,,,Dsec female TW rep2,"SAMN03460235,GSM1650072",,,,,,,,,,,15/07/2026,"About 300-700 pairs of fly antennae of each sex from each species wer

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Adult antenna']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0000972'
library.loc[:,'anatName'] = 'antenna'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX978699,SRP056883,Illumina HiSeq 2000,SRS895369,UBERON:0000972,antenna,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650073,Adult antenna,within 1hr of eclosion,perfect match,not documented,,F,,"Tuson, #14021-0248-30",7238,,,,,,Dsec female TW rep3,"SAMN03460236,GSM1650073",,,,,,,,,,,15/07/2026,"About 300-700 pairs of fly antennae of each sex from each species were collected for total RNA isolation. The antennae resected each day were preserved in ~ 50-100 ul Trizol (Life Technology) and stored at -80 °C before further processing. Right before the RNA isolation, we spun down antennae preserved in Trizol at the max speed for 3 min at 4 °C and put each sample into one MagNA Lyser Green Beads tube (Roche). The RNA isolation protocol we developed combined steps from conventional Trizol extraction and RNeasy Micro Elute Kit (QIAGEN, Inc.) with some modifications to obtain high yield and quality of total RNA. Fly antennae were disrupted with MagNA Lyser (Roche) at 7000 rpm for 15 seconds each time and repeated for 4-5 times until the tissue was almost invisible. After 15 sec, we cooled down the lysate on ice for 10 sec to prevent RNA degradation. Tissue lysate was transferred to a new RNase free Eppendorf tube. We used about 400 ul Trizol to rinse the beads of each sample and combine this 400 ul Trizol with the tissue lysate for the RNA isolation. Based on the suggestion of the Trizol standard protocol, 200 microliters of chloroform per microliter of tissue lysate were added to the lysate and mixed well by shaking vigorously for 15 sec and set 2-3 min at room temperature. The aqueous phase was separated by centrifuging the lysate at the maximum speed at 4 °C for 15 min and carefully transferred to a new Eppendorf tube after centrifuging. We added 1 volume of 5 µl Carrier RNA (QIAGEN, Inc.) and one microliter of 70% EtOH to the aqueous phase and mixed well by inverting the tubes carefully. To obtain greater amount of RNA, we kept the samples at -80 °C for overnight to aid in RNA precipitation. RNA isolation followed manufacturer’s protocol of the RNeasy Micro Elute Kit with increased volumes (700 ul) of RPE buffer and 80% ethanol to wash RNA. Genomic DNA was removed by applying on-column DNase I treatment at room temperature for 15 min. For paired-end mRNA-seq library preparation, we used Illumina TrueSeq kits. A total of 10 µg total RNA was used for mRNA enrichment by oligo-dT beads followed by cation-catalyzed fragmentation for 4 minutes at 94˚C. The mRNA fragments were then converted into double stranded cDNA by random priming followed by end repair. The fragmented cDNAs from each sex of each species were then ligated to the barcoded paired-end adaptors and subjected to 15 cycles of PCR amplification and purified by Ampure beads (Beckman Agencourt). The absolute concentrations of the libraries were determined by Qubit fluorometry (Invitrogen) and BioAnalyzer High Sensitivity DNA Kit (Agilent). We combined barcoded cDNA from 6 samples and loaded them in 3 sequencing lanes of flow cell, and paired-end 2*100nt sequencing was conducted on Illumina HiSeq2000.",,,,Adult antenna,,Adult,,TRANSCRIPTOMIC,cDNA
1,SRX978698,SRP056883,Illumina HiSeq 2000,SRS895370,UBERON:0000972,antenna,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650072,Adult antenna,within 1hr of eclosion,perfect match,not documented,,F,,"Tuson, #14021-0248-29",7238,,,,,,Dsec female TW rep2,"SAMN03460235,GSM1650

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['within 1hr of eclosion']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000066'
library.loc[:,'stageName'] = 'fully formed stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX978699,SRP056883,Illumina HiSeq 2000,SRS895369,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650073,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,F,,"Tuson, #14021-0248-30",7238,,,,,,Dsec female TW rep3,"SAMN03460236,GSM1650073",,,,,,,,,,,15/07/2026,"About 300-700 pairs of fly antennae of each sex from each species were collected for total RNA isolation. The antennae resected each day were preserved in ~ 50-100 ul Trizol (Life Technology) and stored at -80 °C before further processing. Right before the RNA isolation, we spun down antennae preserved in Trizol at the max speed for 3 min at 4 °C and put each sample into one MagNA Lyser Green Beads tube (Roche). The RNA isolation protocol we developed combined steps from conventional Trizol extraction and RNeasy Micro Elute Kit (QIAGEN, Inc.) with some modifications to obtain high yield and quality of total RNA. Fly antennae were disrupted with MagNA Lyser (Roche) at 7000 rpm for 15 seconds each time and repeated for 4-5 times until the tissue was almost invisible. After 15 sec, we cooled down the lysate on ice for 10 sec to prevent RNA degradation. Tissue lysate was transferred to a new RNase free Eppendorf tube. We used about 400 ul Trizol to rinse the beads of each sample and combine this 400 ul Trizol with the tissue lysate for the RNA isolation. Based on the suggestion of the Trizol standard protocol, 200 microliters of chloroform per microliter of tissue lysate were added to the lysate and mixed well by shaking vigorously for 15 sec and set 2-3 min at room temperature. The aqueous phase was separated by centrifuging the lysate at the maximum speed at 4 °C for 15 min and carefully transferred to a new Eppendorf tube after centrifuging. We added 1 volume of 5 µl Carrier RNA (QIAGEN, Inc.) and one microliter of 70% EtOH to the aqueous phase and mixed well by inverting the tubes carefully. To obtain greater amount of RNA, we kept the samples at -80 °C for overnight to aid in RNA precipitation. RNA isolation followed manufacturer’s protocol of the RNeasy Micro Elute Kit with increased volumes (700 ul) of RPE buffer and 80% ethanol to wash RNA. Genomic DNA was removed by applying on-column DNase I treatment at room temperature for 15 min. For paired-end mRNA-seq library preparation, we used Illumina TrueSeq kits. A total of 10 µg total RNA was used for mRNA enrichment by oligo-dT beads followed by cation-catalyzed fragmentation for 4 minutes at 94˚C. The mRNA fragments were then converted into double stranded cDNA by random priming followed by end repair. The fragmented cDNAs from each sex of each species were then ligated to the barcoded paired-end adaptors and subjected to 15 cycles of PCR amplification and purified by Ampure beads (Beckman Agencourt). The absolute concentrations of the libraries were determined by Qubit fluorometry (Invitrogen) and BioAnalyzer High Sensitivity DNA Kit (Agilent). We combined barcoded cDNA from 6 samples and loaded them in 3 sequencing lanes of flow cell, and paired-end 2*100nt sequencing was conducted on Illumina HiSeq2000.",,,,Adult antenna,,Adult,,TRANSCRIPTOMIC,cDNA
1,SRX978698,SRP056883,Illumina HiSeq 2000,SRS895370,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650072,Adult antenna,within 1hr of eclosion,perfect match,not documented,pe

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [16]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

library.loc[:,'strain'] = 'Rob3c / Tucson 14021-0248.25'

library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX978699,SRP056883,Illumina HiSeq 2000,SRS895369,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650073,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,F,Rob3c / Tucson 14021-0248.25,,7238,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Dsec female TW rep3,"SAMN03460236,GSM1650073",1,hour post eclosion,,,,,PMID: 26430061,,,SAC,2026-07-16,"About 300-700 pairs of fly antennae of each sex from each species were collected for total RNA isolation. The antennae resected each day were preserved in ~ 50-100 ul Trizol (Life Technology) and stored at -80 °C before further processing. Right before the RNA isolation, we spun down antennae preserved in Trizol at the max speed for 3 min at 4 °C and put each sample into one MagNA Lyser Green Beads tube (Roche). The RNA isolation protocol we developed combined steps from conventional Trizol extraction and RNeasy Micro Elute Kit (QIAGEN, Inc.) with some modifications to obtain high yield and quality of total RNA. Fly antennae were disrupted with MagNA Lyser (Roche) at 7000 rpm for 15 seconds each time and repeated for 4-5 times until the tissue was almost invisible. After 15 sec, we cooled down the lysate on ice for 10 sec to prevent RNA degradation. Tissue lysate was transferred to a new RNase free Eppendorf tube. We used about 400 ul Trizol to rinse the beads of each sample and combine this 400 ul Trizol with the tissue lysate for the RNA isolation. Based on the suggestion of the Trizol standard protocol, 200 microliters of chloroform per microliter of tissue lysate were added to the lysate and mixed well by shaking vigorously for 15 sec and set 2-3 min at room temperature. The aqueous phase was separated by centrifuging the lysate at the maximum speed at 4 °C for 15 min and carefully transferred to a new Eppendorf tube after centrifuging. We added 1 volume of 5 µl Carrier RNA (QIAGEN, Inc.) and one microliter of 70% EtOH to the aqueous phase and mixed well by inverting the tubes carefully. To obtain greater amount of RNA, we kept the samples at -80 °C for overnight to aid in RNA precipitation. RNA isolation followed manufacturer’s protocol of the RNeasy Micro Elute Kit with increased volumes (700 ul) of RPE buffer and 80% ethanol to wash RNA. Genomic DNA was removed by applying on-column DNase I treatment at room temperature for 15 min. For paired-end mRNA-seq library preparation, we used Illumina TrueSeq kits. A total of 10 µg total RNA was used for mRNA enrichment by oligo-dT beads followed by cation-catalyzed fragmentation for 4 minutes at 94˚C. The mRNA fragments were then converted into double stranded cDNA by random priming followed by end repair. The fragmented cDNAs from each sex of each species were then ligated to the barcoded paired-end adaptors and subjected to 15 cycles of PCR amplification and purified by Ampure beads (Beckman Agencourt). The absolute concentrations of the libraries were determined by Qubit fluorometry (Invitrogen) and BioAnalyzer High Sensitivity DNA Kit (Agilent). We combined barcoded cDNA from 6 samples and loaded them in 3 sequencing lanes of flow cell, and paired-end 2*100nt sequencing was conducted on Illumina HiSeq2000.",,,,Adult antenna,,Adult,,TRANSCRIPTOMIC,cDNA
1,SRX978698,SRP056883,Illumina HiSeq 2000,SRS895370,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/a

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [10]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq RNA Sample Preparation Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX978699,SRP056883,Illumina HiSeq 2000,SRS895369,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650073,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,F,Rob3c / Tucson 14021-0248.25,"Tuson, #14021-0248-30",7238,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Dsec female TW rep3,"SAMN03460236,GSM1650073",,,,,,,,,,,15/07/2026,"About 300-700 pairs of fly antennae of each sex from each species were collected for total RNA isolation. The antennae resected each day were preserved in ~ 50-100 ul Trizol (Life Technology) and stored at -80 °C before further processing. Right before the RNA isolation, we spun down antennae preserved in Trizol at the max speed for 3 min at 4 °C and put each sample into one MagNA Lyser Green Beads tube (Roche). The RNA isolation protocol we developed combined steps from conventional Trizol extraction and RNeasy Micro Elute Kit (QIAGEN, Inc.) with some modifications to obtain high yield and quality of total RNA. Fly antennae were disrupted with MagNA Lyser (Roche) at 7000 rpm for 15 seconds each time and repeated for 4-5 times until the tissue was almost invisible. After 15 sec, we cooled down the lysate on ice for 10 sec to prevent RNA degradation. Tissue lysate was transferred to a new RNase free Eppendorf tube. We used about 400 ul Trizol to rinse the beads of each sample and combine this 400 ul Trizol with the tissue lysate for the RNA isolation. Based on the suggestion of the Trizol standard protocol, 200 microliters of chloroform per microliter of tissue lysate were added to the lysate and mixed well by shaking vigorously for 15 sec and set 2-3 min at room temperature. The aqueous phase was separated by centrifuging the lysate at the maximum speed at 4 °C for 15 min and carefully transferred to a new Eppendorf tube after centrifuging. We added 1 volume of 5 µl Carrier RNA (QIAGEN, Inc.) and one microliter of 70% EtOH to the aqueous phase and mixed well by inverting the tubes carefully. To obtain greater amount of RNA, we kept the samples at -80 °C for overnight to aid in RNA precipitation. RNA isolation followed manufacturer’s protocol of the RNeasy Micro Elute Kit with increased volumes (700 ul) of RPE buffer and 80% ethanol to wash RNA. Genomic DNA was removed by applying on-column DNase I treatment at room temperature for 15 min. For paired-end mRNA-seq library preparation, we used Illumina TrueSeq kits. A total of 10 µg total RNA was used for mRNA enrichment by oligo-dT beads followed by cation-catalyzed fragmentation for 4 minutes at 94˚C. The mRNA fragments were then converted into double stranded cDNA by random priming followed by end repair. The fragmented cDNAs from each sex of each species were then ligated to the barcoded paired-end adaptors and subjected to 15 cycles of PCR amplification and purified by Ampure beads (Beckman Agencourt). The absolute concentrations of the libraries were determined by Qubit fluorometry (Invitrogen) and BioAnalyzer High Sensitivity DNA Kit (Agilent). We combined barcoded cDNA from 6 samples and loaded them in 3 sequencing lanes of flow cell, and paired-end 2*100nt sequencing was conducted on Illumina HiSeq2000.",,,,Adult antenna,,Adult,,TRANSCRIPTOMIC,cDNA
1,SRX978698,SRP056883,Illumina HiSeq 2000,SRS895370,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GS

#### globin, replicates

In [11]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [12]:
library.loc[:,'sampleAge_value'] = '1'
library.loc[:,'sampleAge_unit'] = 'hour post eclosion'

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX978699,SRP056883,Illumina HiSeq 2000,SRS895369,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650073,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,F,Rob3c / Tucson 14021-0248.25,"Tuson, #14021-0248-30",7238,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Dsec female TW rep3,"SAMN03460236,GSM1650073",1,hour post eclosion,,,,,,,,,15/07/2026,"About 300-700 pairs of fly antennae of each sex from each species were collected for total RNA isolation. The antennae resected each day were preserved in ~ 50-100 ul Trizol (Life Technology) and stored at -80 °C before further processing. Right before the RNA isolation, we spun down antennae preserved in Trizol at the max speed for 3 min at 4 °C and put each sample into one MagNA Lyser Green Beads tube (Roche). The RNA isolation protocol we developed combined steps from conventional Trizol extraction and RNeasy Micro Elute Kit (QIAGEN, Inc.) with some modifications to obtain high yield and quality of total RNA. Fly antennae were disrupted with MagNA Lyser (Roche) at 7000 rpm for 15 seconds each time and repeated for 4-5 times until the tissue was almost invisible. After 15 sec, we cooled down the lysate on ice for 10 sec to prevent RNA degradation. Tissue lysate was transferred to a new RNase free Eppendorf tube. We used about 400 ul Trizol to rinse the beads of each sample and combine this 400 ul Trizol with the tissue lysate for the RNA isolation. Based on the suggestion of the Trizol standard protocol, 200 microliters of chloroform per microliter of tissue lysate were added to the lysate and mixed well by shaking vigorously for 15 sec and set 2-3 min at room temperature. The aqueous phase was separated by centrifuging the lysate at the maximum speed at 4 °C for 15 min and carefully transferred to a new Eppendorf tube after centrifuging. We added 1 volume of 5 µl Carrier RNA (QIAGEN, Inc.) and one microliter of 70% EtOH to the aqueous phase and mixed well by inverting the tubes carefully. To obtain greater amount of RNA, we kept the samples at -80 °C for overnight to aid in RNA precipitation. RNA isolation followed manufacturer’s protocol of the RNeasy Micro Elute Kit with increased volumes (700 ul) of RPE buffer and 80% ethanol to wash RNA. Genomic DNA was removed by applying on-column DNase I treatment at room temperature for 15 min. For paired-end mRNA-seq library preparation, we used Illumina TrueSeq kits. A total of 10 µg total RNA was used for mRNA enrichment by oligo-dT beads followed by cation-catalyzed fragmentation for 4 minutes at 94˚C. The mRNA fragments were then converted into double stranded cDNA by random priming followed by end repair. The fragmented cDNAs from each sex of each species were then ligated to the barcoded paired-end adaptors and subjected to 15 cycles of PCR amplification and purified by Ampure beads (Beckman Agencourt). The absolute concentrations of the libraries were determined by Qubit fluorometry (Invitrogen) and BioAnalyzer High Sensitivity DNA Kit (Agilent). We combined barcoded cDNA from 6 samples and loaded them in 3 sequencing lanes of flow cell, and paired-end 2*100nt sequencing was conducted on Illumina HiSeq2000.",,,,Adult antenna,,Adult,,TRANSCRIPTOMIC,cDNA
1,SRX978698,SRP056883,Illumina HiSeq 2000,SRS895370,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/q

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [13]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-07-16'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX978699,SRP056883,Illumina HiSeq 2000,SRS895369,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650073,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,F,Rob3c / Tucson 14021-0248.25,"Tuson, #14021-0248-30",7238,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Dsec female TW rep3,"SAMN03460236,GSM1650073",1,hour post eclosion,,,,,,,,SAC,2026-07-16,"About 300-700 pairs of fly antennae of each sex from each species were collected for total RNA isolation. The antennae resected each day were preserved in ~ 50-100 ul Trizol (Life Technology) and stored at -80 °C before further processing. Right before the RNA isolation, we spun down antennae preserved in Trizol at the max speed for 3 min at 4 °C and put each sample into one MagNA Lyser Green Beads tube (Roche). The RNA isolation protocol we developed combined steps from conventional Trizol extraction and RNeasy Micro Elute Kit (QIAGEN, Inc.) with some modifications to obtain high yield and quality of total RNA. Fly antennae were disrupted with MagNA Lyser (Roche) at 7000 rpm for 15 seconds each time and repeated for 4-5 times until the tissue was almost invisible. After 15 sec, we cooled down the lysate on ice for 10 sec to prevent RNA degradation. Tissue lysate was transferred to a new RNase free Eppendorf tube. We used about 400 ul Trizol to rinse the beads of each sample and combine this 400 ul Trizol with the tissue lysate for the RNA isolation. Based on the suggestion of the Trizol standard protocol, 200 microliters of chloroform per microliter of tissue lysate were added to the lysate and mixed well by shaking vigorously for 15 sec and set 2-3 min at room temperature. The aqueous phase was separated by centrifuging the lysate at the maximum speed at 4 °C for 15 min and carefully transferred to a new Eppendorf tube after centrifuging. We added 1 volume of 5 µl Carrier RNA (QIAGEN, Inc.) and one microliter of 70% EtOH to the aqueous phase and mixed well by inverting the tubes carefully. To obtain greater amount of RNA, we kept the samples at -80 °C for overnight to aid in RNA precipitation. RNA isolation followed manufacturer’s protocol of the RNeasy Micro Elute Kit with increased volumes (700 ul) of RPE buffer and 80% ethanol to wash RNA. Genomic DNA was removed by applying on-column DNase I treatment at room temperature for 15 min. For paired-end mRNA-seq library preparation, we used Illumina TrueSeq kits. A total of 10 µg total RNA was used for mRNA enrichment by oligo-dT beads followed by cation-catalyzed fragmentation for 4 minutes at 94˚C. The mRNA fragments were then converted into double stranded cDNA by random priming followed by end repair. The fragmented cDNAs from each sex of each species were then ligated to the barcoded paired-end adaptors and subjected to 15 cycles of PCR amplification and purified by Ampure beads (Beckman Agencourt). The absolute concentrations of the libraries were determined by Qubit fluorometry (Invitrogen) and BioAnalyzer High Sensitivity DNA Kit (Agilent). We combined barcoded cDNA from 6 samples and loaded them in 3 sequencing lanes of flow cell, and paired-end 2*100nt sequencing was conducted on Illumina HiSeq2000.",,,,Adult antenna,,Adult,,TRANSCRIPTOMIC,cDNA
1,SRX978698,SRP056883,Illumina HiSeq 2000,SRS895370,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/ge

#### comments

In [14]:
library.loc[:,'comment'] = 'PMID: 26430061'

#### save complete file with correct columns

In [17]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX978699,SRP056883,Illumina HiSeq 2000,SRS895369,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650073,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,F,Rob3c / Tucson 14021-0248.25,,7238,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Dsec female TW rep3,"SAMN03460236,GSM1650073",1,hour post eclosion,,,,,PMID: 26430061,,,SAC,2026-07-16
1,SRX978698,SRP056883,Illumina HiSeq 2000,SRS895370,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650072,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,F,Rob3c / Tucson 14021-0248.25,,7238,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Dsec female TW rep2,"SAMN03460235,GSM1650072",1,hour post eclosion,,,,,PMID: 26430061,,,SAC,2026-07-16
2,SRX978697,SRP056883,Illumina HiSeq 2000,SRS895371,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650071,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,F,Rob3c / Tucson 14021-0248.25,,7238,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Dsec female TW rep1,"SAMN03460234,GSM1650071",1,hour post eclosion,,,,,PMID: 26430061,,,SAC,2026-07-16
3,SRX978696,SRP056883,Illumina HiSeq 2000,SRS895372,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650070,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,M,Rob3c / Tucson 14021-0248.25,,7238,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Dsec male TW rep3,"SAMN03460233,GSM1650070",1,hour post eclosion,,,,,PMID: 26430061,,,SAC,2026-07-16
4,SRX978695,SRP056883,Illumina HiSeq 2000,SRS895373,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650069,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,M,Rob3c / Tucson 14021-0248.25,,7238,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Dsec male TW rep2,"SAMN03460232,GSM1650069",1,hour post eclosion,,,,,PMID: 26430061,,,SAC,2026-07-16
5,SRX978694,SRP056883,Illumina HiSeq 2000,SRS895374,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1650068,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,M,Rob3c / Tucson 14021-0248.25,,7238,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Dsec male TW rep1,"SAMN03460231,GSM1650068",1,hour post eclosion,,,,,PMID: 26430061,,,SAC,2026-07-16


### experiment annotations

In [18]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP056883,Expression divergence of chemosensory genes between Drosophila sechellia and its sibling species and its implications for host shift [Dsec TW],"Drosophila sechellia relies exclusively on the fruits of Morinda citrifolia, which are toxic to most insects, including its sibling species D. melanogaster and D. simulans. Although several odorant binding protein (Obp) genes and olfactory receptor (Or) genes were suggested to be associated with the D. sechellia host shift, a broad view of how chemosensory genes have contributed to this shift is still lacking. We therefore studied the antennal transcriptomes, the main organ responsible for detecting food resource and oviposition, of D. sechellia and its two sibling species. We wanted to know whether gene expression, particularly chemosensory genes, has diverged between D. sechellia and its two sibling species. Using a very stringent definition of differential gene expression, we found 147 genes (including 11 chemosensory genes) were up-regulated while only 81 genes (including 5 chemosensory genes) were down-regulated in D. sechellia. Interestingly, Obp50a exhibited the highest up-regulation, a ~100 fold increase, and Or85c – previously reported to be a larva-specific gene– showed ~20 fold up-regulation in D. sechellia. Furthermore, Ir84a, proposed to be associated with male courtship behavior, is significantly up-regulated in D. sechellia. We also found expression divergence in most of the receptor gene families between D. sechellia and the two sibling species. Our observations suggest that the host shift of D. sechellia is associated with expression profile divergence in all chemosensory gene families and is achieved mostly by up-regulation of chemosensory genes. Overall design: RNAseq experiments in wild type drosophila antennaes. The strain is D. sechellia (Tuson, #14021-0248-25).",SRA,,,,,,GSE67587,PRJNA280377,26430061,,10.1093/gbe/evv183,,


#### experiment and protocol details

In [19]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

6

In [20]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP056883,Expression divergence of chemosensory genes between Drosophila sechellia and its sibling species and its implications for host shift [Dsec TW],"Drosophila sechellia relies exclusively on the fruits of Morinda citrifolia, which are toxic to most insects, including its sibling species D. melanogaster and D. simulans. Although several odorant binding protein (Obp) genes and olfactory receptor (Or) genes were suggested to be associated with the D. sechellia host shift, a broad view of how chemosensory genes have contributed to this shift is still lacking. We therefore studied the antennal transcriptomes, the main organ responsible for detecting food resource and oviposition, of D. sechellia and its two sibling species. We wanted to know whether gene expression, particularly chemosensory genes, has diverged between D. sechellia and its two sibling species. Using a very stringent definition of differential gene expression, we found 147 genes (including 11 chemosensory genes) were up-regulated while only 81 genes (including 5 chemosensory genes) were down-regulated in D. sechellia. Interestingly, Obp50a exhibited the highest up-regulation, a ~100 fold increase, and Or85c – previously reported to be a larva-specific gene– showed ~20 fold up-regulation in D. sechellia. Furthermore, Ir84a, proposed to be associated with male courtship behavior, is significantly up-regulated in D. sechellia. We also found expression divergence in most of the receptor gene families between D. sechellia and the two sibling species. Our observations suggest that the host shift of D. sechellia is associated with expression profile divergence in all chemosensory gene families and is achieved mostly by up-regulation of chemosensory genes. Overall design: RNAseq experiments in wild type drosophila antennaes. The strain is D. sechellia (Tuson, #14021-0248-25).",SRA,total,Bgee 1K,6,TruSeq RNA Sample Preparation Kit,full_length,GSE67587,PRJNA280377,26430061,,10.1093/gbe/evv183,,


#### paper and xrefs

In [21]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '26430061'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC4684695/'
experiment.loc[:,'DOI'] = '10.1093/gbe/evv183'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP056883,Expression divergence of chemosensory genes between Drosophila sechellia and its sibling species and its implications for host shift [Dsec TW],"Drosophila sechellia relies exclusively on the fruits of Morinda citrifolia, which are toxic to most insects, including its sibling species D. melanogaster and D. simulans. Although several odorant binding protein (Obp) genes and olfactory receptor (Or) genes were suggested to be associated with the D. sechellia host shift, a broad view of how chemosensory genes have contributed to this shift is still lacking. We therefore studied the antennal transcriptomes, the main organ responsible for detecting food resource and oviposition, of D. sechellia and its two sibling species. We wanted to know whether gene expression, particularly chemosensory genes, has diverged between D. sechellia and its two sibling species. Using a very stringent definition of differential gene expression, we found 147 genes (including 11 chemosensory genes) were up-regulated while only 81 genes (including 5 chemosensory genes) were down-regulated in D. sechellia. Interestingly, Obp50a exhibited the highest up-regulation, a ~100 fold increase, and Or85c – previously reported to be a larva-specific gene– showed ~20 fold up-regulation in D. sechellia. Furthermore, Ir84a, proposed to be associated with male courtship behavior, is significantly up-regulated in D. sechellia. We also found expression divergence in most of the receptor gene families between D. sechellia and the two sibling species. Our observations suggest that the host shift of D. sechellia is associated with expression profile divergence in all chemosensory gene families and is achieved mostly by up-regulation of chemosensory genes. Overall design: RNAseq experiments in wild type drosophila antennaes. The strain is D. sechellia (Tuson, #14021-0248-25).",SRA,total,Bgee 1K,6,TruSeq RNA Sample Preparation Kit,full_length,GSE67587,PRJNA280377,26430061,https://pmc.ncbi.nlm.nih.gov/articles/PMC4684695/,10.1093/gbe/evv183,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [22]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [23]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [24]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 6
Errors: 6
Warnings: 0
Top codes:
  - BAD_VALUE: 6


#### check columns match

In [28]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [29]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
68054,SRX2004068,SRP080993,Illumina HiSeq 2500,SRS1603636,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Antenna,Adult,perfect match,not documented,perfect match,mixed,14024-0371.00,,7217,TruSeq Stranded mRNA,full_length,polyA,,,ana_AA_2,"SAMN05513846,GSM2262741",7-10,day post eclosion,,,,,PMID: 28821769,,,SAC,2026-07-15
68055,SRX2004067,SRP080993,Illumina HiSeq 2500,SRS1603635,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Antenna,Adult,perfect match,not documented,perfect match,mixed,14024-0371.00,,7217,TruSeq Stranded mRNA,full_length,polyA,,,ana_AA_1,"SAMN05513847,GSM2262740",7-10,day post eclosion,,,,,PMID: 28821769,,,SAC,2026-07-15
68056,SRX978699,SRP056883,Illumina HiSeq 2000,SRS895369,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,F,Rob3c / Tucson 14021-0248.25,,7238,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Dsec female TW rep3,"SAMN03460236,GSM1650073",1,hour post eclosion,,,,,PMID: 26430061,,,SAC,2026-07-16
68057,SRX978698,SRP056883,Illumina HiSeq 2000,SRS895370,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,F,Rob3c / Tucson 14021-0248.25,,7238,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Dsec female TW rep2,"SAMN03460235,GSM1650072",1,hour post eclosion,,,,,PMID: 26430061,,,SAC,2026-07-16
68058,SRX978697,SRP056883,Illumina HiSeq 2000,SRS895371,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,F,Rob3c / Tucson 14021-0248.25,,7238,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Dsec female TW rep1,"SAMN03460234,GSM1650071",1,hour post eclosion,,,,,PMID: 26430061,,,SAC,2026-07-16
68059,SRX978696,SRP056883,Illumina HiSeq 2000,SRS895372,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,M,Rob3c / Tucson 14021-0248.25,,7238,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Dsec male TW rep3,"SAMN03460233,GSM1650070",1,hour post eclosion,,,,,PMID: 26430061,,,SAC,2026-07-16
68060,SRX978695,SRP056883,Illumina HiSeq 2000,SRS895373,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Adult antenna,within 1hr of eclosion,perfect match,not documented,perfect match,M,Rob3c / Tucson 14021-0248.25,,7238,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Dsec male TW rep2,"SAMN03460232,GSM1650069",1,hour post eclosion,,,,,PMID: 26430061,,,SAC,2026-07-16


In [30]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1290,SRP219656,Evolved differences in cis and trans regulatio...,"During embryogenesis in animals, initial devel...",SRA,total,Bgee 1K,63,TruSeq Stranded mRNA,full_length,GSE136646,PRJNA562920,32928902,https://pmc.ncbi.nlm.nih.gov/articles/PMC7648588/,10.1534/genetics.120.303626,,
1291,SRP080993,Developmental hotspots drive transcriptional v...,The goal of this study was to identify the pre...,SRA,total,Bgee 1K,40,TruSeq Stranded mRNA,full_length,GSE85239,PRJNA337934,28821769,https://pmc.ncbi.nlm.nih.gov/articles/PMC5562767/,10.1038/s41598-017-08563-0,28644712[PMID],
1292,SRP056883,Expression divergence of chemosensory genes be...,Drosophila sechellia relies exclusively on the...,SRA,total,Bgee 1K,6,TruSeq RNA Sample Preparation Kit,full_length,GSE67587,PRJNA280377,26430061,https://pmc.ncbi.nlm.nih.gov/articles/PMC4684695/,10.1093/gbe/evv183,,


### add annotations to git

In [31]:
! git pull

Already up to date.


In [32]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [33]:
! git add $git_experiment_path $git_library_path

In [34]:
! git commit -m $commit_message_exp

[develop ba891d2] adding annotated bulk experiment SRP056883
 2 files changed, 7 insertions(+)


In [35]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.94 KiB | 753.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   7b1f94b..ba891d2  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push